In [99]:
import ccxt
import pandas as pd
import numpy as np
import ta
import time

from colorama import Fore, Style, init
from tabulate import tabulate

init(autoreset=True)

In [100]:
exchange = ccxt.binance({
    'enableRateLimit': True,
    'options': {
        'defaultType': 'future'
    }
})

In [101]:
# ============================================================
# SETTINGS
# ============================================================

TIMEFRAME = '15m'
LIMIT = 250

TOP_VOLUME_COINS = 80

MIN_24H_VOLUME = 50_000_000

ATR_MULTIPLIER = 1.2

VOLUME_SPIKE_MULTIPLIER = 1.2

DISPLACEMENT_MULTIPLIER = 1.2

MIN_VOLATILITY_PERCENT = 1.0

SLEEP_TIME = 0.1

In [102]:
# ============================================================
# FETCH FUTURES USDT PAIRS
# ============================================================

def fetch_usdt_pairs():

    print(Fore.CYAN + "\nLoading Binance Futures Markets...\n")

    markets = exchange.load_markets()

    symbols = []

    for symbol in markets:

        market = markets[symbol]

        try:

            if (
                market['quote'] == 'USDT'
                and market['active']
                and market['contract']
            ):

                symbols.append(symbol)

        except:
            pass

    print(Fore.GREEN + f"Found {len(symbols)} USDT Futures Pairs\n")

    return symbols

In [103]:
# ============================================================
# FETCH TICKERS
# ============================================================

def fetch_top_volume_symbols(symbols):

    print(Fore.CYAN + "Filtering High Volume Coins...\n")

    tickers = exchange.fetch_tickers()

    filtered = []

    for symbol in symbols:

        try:

            ticker = tickers[symbol]

            quote_volume = ticker['quoteVolume']

            if quote_volume is None:
                continue

            if quote_volume >= MIN_24H_VOLUME:

                filtered.append({
                    'symbol': symbol,
                    'quoteVolume': quote_volume
                })

        except:
            pass

    filtered = sorted(
        filtered,
        key=lambda x: x['quoteVolume'],
        reverse=True
    )

    filtered = filtered[:TOP_VOLUME_COINS]

    final_symbols = [x['symbol'] for x in filtered]

    print(
        Fore.GREEN
        + f"Selected {len(final_symbols)} High Volume Coins\n"
    )

    return final_symbols

In [104]:
# ============================================================
# FETCH OHLCV DATA
# ============================================================

def fetch_ohlcv(symbol):

    try:

        ohlcv = exchange.fetch_ohlcv(
            symbol=symbol,
            timeframe=TIMEFRAME,
            limit=LIMIT
        )

        df = pd.DataFrame(
            ohlcv,
            columns=[
                'timestamp',
                'open',
                'high',
                'low',
                'close',
                'volume'
            ]
        )

        df['timestamp'] = pd.to_datetime(
            df['timestamp'],
            unit='ms'
        )

        return df

    except Exception as e:

        print(
            Fore.RED
            + f"Error Fetching {symbol}: {e}"
        )

        return None


In [105]:
# ============================================================
# ADD MOVING AVERAGES
# ============================================================

def add_moving_averages(df):

    # EMA WORKS BETTER FOR CRYPTO

    df['ma7'] = ta.trend.ema_indicator(
        close=df['close'],
        window=7
    )

    df['ma25'] = ta.trend.ema_indicator(
        close=df['close'],
        window=25
    )

    df['ma99'] = ta.trend.ema_indicator(
        close=df['close'],
        window=99
    )

    return df


In [106]:
# ============================================================
# ADD ATR
# ============================================================

def add_atr(df):

    atr_indicator = ta.volatility.AverageTrueRange(
        high=df['high'],
        low=df['low'],
        close=df['close'],
        window=14
    )

    df['atr'] = atr_indicator.average_true_range()

    df['atr_average'] = (
        df['atr']
        .rolling(14)
        .mean()
    )

    return df

In [107]:
# ============================================================
# ADD VOLATILITY
# ============================================================

def add_volatility(df):

    highest_price = (
        df['high']
        .rolling(20)
        .max()
    )

    lowest_price = (
        df['low']
        .rolling(20)
        .min()
    )

    df['volatility_percent'] = (
        (
            highest_price - lowest_price
        ) / df['close']
    ) * 100

    return df

In [108]:
# ============================================================
# ADD VOLUME LOGIC
# ============================================================

def add_volume_analysis(df):

    df['volume_avg'] = (
        df['volume']
        .rolling(20)
        .mean()
    )

    df['relative_volume'] = (
        df['volume']
        / df['volume_avg']
    )

    return df


In [109]:
# ============================================================
# DETECT CONSOLIDATION
# ============================================================

def detect_consolidation(df):

    recent_range = (
        df['high'].rolling(10).max()
        - df['low'].rolling(10).min()
    )

    avg_range = (
        (df['high'] - df['low'])
        .rolling(20)
        .mean()
    )

    latest_range = recent_range.iloc[-1]

    latest_avg = avg_range.iloc[-1]

    consolidation = (
        latest_range < latest_avg * 2
    )

    return consolidation

In [110]:
# ============================================================
# DETECT DISPLACEMENT CANDLE
# ============================================================

def detect_displacement(df):

    latest = df.iloc[-1]

    candle_range = (
        latest['high']
        - latest['low']
    )

    avg_range = (
        (df['high'] - df['low'])
        .rolling(20)
        .mean()
        .iloc[-1]
    )

    return candle_range > avg_range * DISPLACEMENT_MULTIPLIER


In [111]:
def detect_early_expansion(df):

    latest = df.iloc[-1]

    previous = df.iloc[-2]

    score = 0

    # ====================================================
    # EMA TREND PRESSURE
    # ====================================================

    if latest['ma7'] > latest['ma25']:
        score += 1

    # ====================================================
    # EMA SLOPE TURNING UP
    # ====================================================

    if latest['ma7'] > previous['ma7']:
        score += 1

    # ====================================================
    # PRICE HOLDING ABOVE EMA
    # ====================================================

    if latest['close'] > latest['ma7']:
        score += 1

    # ====================================================
    # ATR STARTING TO RISE
    # ====================================================

    if latest['atr'] > previous['atr']:
        score += 1

    # ====================================================
    # SLIGHT VOLUME INCREASE
    # ====================================================

    if latest['relative_volume'] > 1.05:
        score += 1

    # ====================================================
    # VOLATILITY STARTING TO EXPAND
    # ====================================================

    if latest['volatility_percent'] > 0.5:
        score += 1

    # ====================================================
    # CONSECUTIVE BULLISH CANDLES
    # ====================================================

    bullish_closes = (
        latest['close'] > latest['open']
        and previous['close'] > previous['open']
    )

    if bullish_closes:
        score += 1

    # ====================================================
    # BEARISH CONDITIONS
    # ====================================================

    bear_score = 0

    if latest['ma7'] < latest['ma25']:
        bear_score += 1

    if latest['ma7'] < previous['ma7']:
        bear_score += 1

    if latest['close'] < latest['ma7']:
        bear_score += 1

    if latest['atr'] > previous['atr']:
        bear_score += 1

    if latest['relative_volume'] > 1.05:
        bear_score += 1

    if latest['volatility_percent'] > 0.5:
        bear_score += 1

    bearish_closes = (
        latest['close'] < latest['open']
        and previous['close'] < previous['open']
    )

    if bearish_closes:
        bear_score += 1

    # ====================================================
    # FINAL SIGNALS
    # ====================================================

    if score >= 5:
        return "EARLY_BULLISH", score

    elif bear_score >= 5:
        return "EARLY_BEARISH", bear_score

    return None, 0

In [112]:
# ============================================================
# SCAN MARKET
# ============================================================

def scan_market():

    print(
        Fore.YELLOW
        + "\nSTARTING INSTITUTIONAL EXPANSION SCANNER\n"
    )

    symbols = fetch_usdt_pairs()

    symbols = fetch_top_volume_symbols(symbols)

    results = []

    for index, symbol in enumerate(symbols):

        try:

            print(
                Fore.CYAN
                + f"[{index+1}/{len(symbols)}] Scanning {symbol}"
            )

            df = fetch_ohlcv(symbol)

            if df is None:
                continue

            # =================================================
            # ADD INDICATORS
            # =================================================

            df = add_moving_averages(df)

            df = add_atr(df)

            df = add_volatility(df)

            df = add_volume_analysis(df)

            # =================================================
            # DETECT SIGNAL
            # =================================================

            signal = detect_early_expansion(df)

            if signal:

                latest = df.iloc[-1]

                results.append({

                    'Symbol': symbol,

                    'Signal': signal,

                    'Price': round(
                        latest['close'],
                        4
                    ),

                    'ATR': round(
                        latest['atr'],
                        4
                    ),

                    'Volatility %': round(
                        latest['volatility_percent'],
                        2
                    ),

                    'Relative Volume': round(
                        latest['relative_volume'],
                        2
                    )
                })

            time.sleep(SLEEP_TIME)

        except Exception as e:

            print(
                Fore.RED
                + f"Scanner Error {symbol}: {e}"
            )

    return results

In [113]:
# ============================================================
# DISPLAY RESULTS
# ============================================================

def display_results(results):

    if len(results) == 0:

        print(
            Fore.RED
            + "\nNO EARLY EXPANSION SETUPS FOUND\n"
        )

        return

    df = pd.DataFrame(results)

    df = df.sort_values(
        by='Relative Volume',
        ascending=False
    )

    print(
        Fore.GREEN
        + "\nEARLY EXPANSION SETUPS FOUND\n"
    )

    print(
        tabulate(
            df,
            headers='keys',
            tablefmt='fancy_grid',
            showindex=False
        )
    )


In [114]:
# ============================================================
# MAIN EXECUTION
# ============================================================

if __name__ == "__main__":

    results = scan_market()

    display_results(results)

    print(
        Fore.CYAN
        + "\nSCAN COMPLETED\n"
    )


STARTING INSTITUTIONAL EXPANSION SCANNER


Loading Binance Futures Markets...

Found 583 USDT Futures Pairs

Filtering High Volume Coins...

Selected 64 High Volume Coins

[1/64] Scanning BTC/USDT:USDT
[2/64] Scanning ETH/USDT:USDT
[3/64] Scanning HYPE/USDT:USDT
[4/64] Scanning SOL/USDT:USDT
[5/64] Scanning ZEC/USDT:USDT
[6/64] Scanning NEAR/USDT:USDT
[7/64] Scanning XAU/USDT:USDT
[8/64] Scanning XAG/USDT:USDT
[9/64] Scanning CL/USDT:USDT
[10/64] Scanning XRP/USDT:USDT
[11/64] Scanning EDEN/USDT:USDT
[12/64] Scanning BEAT/USDT:USDT
[13/64] Scanning BSB/USDT:USDT
[14/64] Scanning DOGE/USDT:USDT
[15/64] Scanning ONDO/USDT:USDT
[16/64] Scanning SUI/USDT:USDT
[17/64] Scanning BNB/USDT:USDT
[18/64] Scanning GENIUS/USDT:USDT
[19/64] Scanning FIDA/USDT:USDT
[20/64] Scanning WLD/USDT:USDT
[21/64] Scanning SNDK/USDT:USDT
[22/64] Scanning BZ/USDT:USDT
[23/64] Scanning PROVE/USDT:USDT
[24/64] Scanning BILL/USDT:USDT
[25/64] Scanning ALT/USDT:USDT
[26/64] Scanning GRASS/USDT:USDT
[27/64] Scanning